<a href="https://colab.research.google.com/github/red-gunslinger/Int-Comp/blob/main/Vector_Stores_y_Busqueda_Semantica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers -q

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import csv

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: np.ndarray | None = None

    def add_documents(self, documents: list[Document]):
        texts = [doc.text for doc in documents]
        new_embeddings = self.model.encode(texts, show_progress_bar=False)
        new_embeddings = np.array(new_embeddings)

        if self.embeddings is None:
            self.embeddings = new_embeddings
            self.documents = list(documents)
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])
            self.documents.extend(documents)

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        query_embedding = self.model.encode([query])
        query_embedding = np.array(query_embedding)

        dot_products = np.dot(self.embeddings, query_embedding.T).flatten()
        doc_norms = np.linalg.norm(self.embeddings, axis=1)
        query_norm = np.linalg.norm(query_embedding)
        similarities = dot_products / (doc_norms * query_norm + 1e-10)

        top_indices = np.argsort(similarities)[::-1][:top_k]

        results = []
        for idx in top_indices:
            results.append(SearchResult(
                score=float(similarities[idx]),
                document=self.documents[idx]
            ))
        return results

In [ ]:
def load_animal_facts(filepath: str) -> list[Document]:
    documents = []
    with open(filepath, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            text = row["text"].strip()
            if not text:
                continue
            metadata = {
                "animal_name": row["animal_name"],
                "source": row["source"],
                "media_link": row["media_link"],
                "wikipedia_link": row["wikipedia_link"],
            }
            documents.append(Document(text=text, metadata=metadata))
    return documents

animal_docs = load_animal_facts("animal-fun-facts-dataset.csv")
print(f"Total documents loaded: {len(animal_docs)}")
print(f"Sample: {animal_docs[0].text[:100]}...")
print(f"Metadata: {animal_docs[0].metadata}")

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

subset = animal_docs[:800]
store = VectorStore(model)
store.add_documents(subset)
print(f"VectorStore created with {len(store.documents)} documents")
print(f"Embedding matrix shape: {store.embeddings.shape}")

In [ ]:
def show_results(query: str, results: list[SearchResult]):
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    for i, r in enumerate(results, 1):
        print(f"\n--- Result {i} (score: {r.score:.4f}) ---")
        print(f"Text: {r.document.text}")
        print(f"Metadata: {r.document.metadata}")

In [ ]:
queries = [
    "animals that live in the ocean",
    "fastest animals in the world",
    "animals with unusual sleeping habits",
    "poisonous or venomous creatures",
    "animals that can fly",
]

for q in queries:
    results = store.search(q, top_k=3)
    show_results(q, results)

In [ ]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: np.ndarray | None = None

    def add_documents(self, documents: list[Document]):
        texts = [doc.text for doc in documents]
        new_embeddings = self.model.encode(texts, show_progress_bar=False)
        new_embeddings = np.array(new_embeddings)

        if self.embeddings is None:
            self.embeddings = new_embeddings
            self.documents = list(documents)
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])
            self.documents.extend(documents)

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        query_embedding = self.model.encode([query])
        query_embedding = np.array(query_embedding)

        dot_products = np.dot(self.embeddings, query_embedding.T).flatten()
        doc_norms = np.linalg.norm(self.embeddings, axis=1)
        query_norm = np.linalg.norm(query_embedding)
        similarities = dot_products / (doc_norms * query_norm + 1e-10)

        if metadata_filter:
            valid_indices = []
            for i, doc in enumerate(self.documents):
                match = all(
                    doc.metadata.get(key) == value
                    for key, value in metadata_filter.items()
                )
                if match:
                    valid_indices.append(i)
            valid_indices = np.array(valid_indices)
            if len(valid_indices) == 0:
                return []
            filtered_sims = similarities[valid_indices]
            top_local = np.argsort(filtered_sims)[::-1][:top_k]
            top_indices = valid_indices[top_local]
        else:
            top_indices = np.argsort(similarities)[::-1][:top_k]

        results = []
        for idx in top_indices:
            results.append(SearchResult(
                score=float(similarities[idx]),
                document=self.documents[idx]
            ))
        return results

In [ ]:
product_reviews = [
    #headphones
    Document("These wireless headphones have incredible noise cancellation that blocks out everything around you. Perfect for flights and commutes.", {"category": "headphones", "brand": "Sony", "sentiment": "positive"}),
    Document("The bass response on these headphones is muddy and the highs are tinny. Not worth the premium price tag at all.", {"category": "headphones", "brand": "Beats", "sentiment": "negative"}),
    Document("Comfortable over-ear design with excellent battery life lasting over 30 hours on a single charge.", {"category": "headphones", "brand": "Sony", "sentiment": "positive"}),
    Document("The Bluetooth connection drops constantly when I walk more than 10 feet from my phone. Very frustrating experience.", {"category": "headphones", "brand": "Beats", "sentiment": "negative"}),
    Document("Studio-quality sound with balanced mids and crystal clear highs. The spatial audio feature is mind-blowing.", {"category": "headphones", "brand": "Apple", "sentiment": "positive"}),
    Document("The ear cushions started peeling after just three months of normal use. Build quality is disappointing.", {"category": "headphones", "brand": "Beats", "sentiment": "negative"}),
    Document("Lightweight design makes these perfect for long listening sessions. The fold-up mechanism is convenient for travel.", {"category": "headphones", "brand": "Bose", "sentiment": "positive"}),
    Document("Active noise cancellation works well on low frequencies but lets through a lot of high-pitched sounds like voices.", {"category": "headphones", "brand": "Bose", "sentiment": "neutral"}),
    # smartphones
    Document("The camera on this phone is absolutely stunning. Night mode produces photos that rival dedicated cameras.", {"category": "smartphone", "brand": "Apple", "sentiment": "positive"}),
    Document("Battery barely lasts half a day with normal usage. I have to carry a power bank everywhere now.", {"category": "smartphone", "brand": "Samsung", "sentiment": "negative"}),
    Document("The 120Hz display is buttery smooth and makes scrolling and animations feel incredibly responsive.", {"category": "smartphone", "brand": "Samsung", "sentiment": "positive"}),
    Document("This phone heats up significantly during video calls and gaming. Thermal management needs improvement.", {"category": "smartphone", "brand": "Google", "sentiment": "negative"}),
    Document("AI-powered photo editing features are a game changer. Magic eraser removes unwanted objects seamlessly.", {"category": "smartphone", "brand": "Google", "sentiment": "positive"}),
    Document("The phone is too large for one-handed use and the weight makes it uncomfortable during long calls.", {"category": "smartphone", "brand": "Samsung", "sentiment": "negative"}),
    Document("Fast charging goes from 0 to 80 percent in under 30 minutes. Wireless charging also works flawlessly.", {"category": "smartphone", "brand": "Apple", "sentiment": "positive"}),
    #laptops
    Document("The M-series chip delivers incredible performance while keeping the laptop completely silent during heavy tasks.", {"category": "laptop", "brand": "Apple", "sentiment": "positive"}),
    Document("The keyboard feels mushy and lacks the tactile feedback I need for comfortable all-day typing sessions.", {"category": "laptop", "brand": "Dell", "sentiment": "negative"}),
    Document("This laptop handles video editing and 3D rendering without breaking a sweat. Export times are impressively fast.", {"category": "laptop", "brand": "Apple", "sentiment": "positive"}),
    Document("The trackpad is too small and frequently registers accidental palm touches while typing documents.", {"category": "laptop", "brand": "Lenovo", "sentiment": "negative"}),
    Document("Fan noise under load is extremely loud, making it impossible to use in quiet environments like libraries.", {"category": "laptop", "brand": "Dell", "sentiment": "negative"}),
    Document("The OLED display has perfect blacks and vibrant colors that make creative work an absolute pleasure.", {"category": "laptop", "brand": "Lenovo", "sentiment": "positive"}),
    Document("Excellent build quality with a solid aluminum chassis that feels premium without being too heavy to carry.", {"category": "laptop", "brand": "Apple", "sentiment": "positive"}),
    #speakers
    Document("Room-filling sound from such a compact speaker is remarkable. The bass is deep and rich for its size.", {"category": "speaker", "brand": "Sonos", "sentiment": "positive"}),
    Document("The voice assistant integration is clunky and often misunderstands basic commands for music playback.", {"category": "speaker", "brand": "Amazon", "sentiment": "negative"}),
    Document("Multi-room audio setup was seamless and the synchronization between speakers is perfect with zero delay.", {"category": "speaker", "brand": "Sonos", "sentiment": "positive"}),
    Document("Waterproof design and rugged build make this the ideal speaker for pool parties and outdoor adventures.", {"category": "speaker", "brand": "JBL", "sentiment": "positive"}),
    Document("The app required for setup is buggy and crashes frequently. Took over an hour to get the speaker connected.", {"category": "speaker", "brand": "Amazon", "sentiment": "negative"}),
    Document("Portable speaker with surprisingly powerful output. The 360-degree sound fills a room evenly from any position.", {"category": "speaker", "brand": "JBL", "sentiment": "positive"}),
    #smartwatches
    Document("Health tracking is comprehensive with heart rate, blood oxygen, sleep stages, and stress monitoring all accurate.", {"category": "smartwatch", "brand": "Apple", "sentiment": "positive"}),
    Document("The screen is nearly impossible to read in direct sunlight even at maximum brightness setting.", {"category": "smartwatch", "brand": "Fitbit", "sentiment": "negative"}),
    Document("GPS tracking for runs is very accurate and the workout detection automatically identifies exercise types.", {"category": "smartwatch", "brand": "Garmin", "sentiment": "positive"}),
    Document("The battery only lasts about 18 hours which means charging every single night without fail.", {"category": "smartwatch", "brand": "Apple", "sentiment": "negative"}),
    Document("Rugged design with sapphire crystal glass and titanium case that survives drops and scratches with ease.", {"category": "smartwatch", "brand": "Garmin", "sentiment": "positive"}),
]

print(f"Product reviews dataset: {len(product_reviews)} documents")
categories = set(d.metadata["category"] for d in product_reviews)
brands = set(d.metadata["brand"] for d in product_reviews)
print(f"Categories: {categories}")
print(f"Brands: {brands}")